<div dir="rtl">

# الدرس 35: الشبكة متعددة الطبقات

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

## لماذا تحتاج الطبقات دالة تفعيل؟

</div>

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
W1 = rng.normal(size=(2, 5))
W2 = rng.normal(size=(5, 3))
X = rng.normal(size=(4, 2))

two_layers = (X @ W1) @ W2
one_layer = X @ (W1 @ W2)
print("same result:", np.allclose(two_layers, one_layer))
print("merged matrix shape:", (W1 @ W2).shape)

<div dir="rtl">

## كل عصبون ريلو يضيف انثناءة

</div>

In [ ]:
x = np.linspace(-3, 3, 300)
target = np.sin(2 * x)

def relu_fit(n_neurons):
    kinks = np.linspace(-3, 3, n_neurons, endpoint=False)
    H = np.maximum(0, x[:, None] - kinks[None, :])   # (300, n)
    H = np.column_stack([np.ones_like(x), H])        # bias column
    w, *_ = np.linalg.lstsq(H, target, rcond=None)
    return H @ w

for n in [3, 6, 20]:
    mse = np.mean((relu_fit(n) - target) ** 2)
    print(f"{n:2} neurons: MSE = {mse:.4f}")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), sharey=True)
for ax, n in zip(axes, [3, 6, 20]):
    ax.plot(x, target, lw=5, alpha=0.25, label="target")
    ax.plot(x, relu_fit(n), lw=2, label="network")
    ax.set_title(f"{n} ReLU neurons"); ax.grid(True)
axes[0].legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

<div dir="rtl">

### التمرير الأمامي في نمباي

</div>

In [ ]:
def relu(z):
    return np.maximum(0, z)

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
W1 = np.array([[1, 1],
               [1, 1]])
b1 = np.array([0, -1])
W2 = np.array([[1],
               [-2]])
b2 = np.array([0])

H = relu(X @ W1 + b1)
out = H @ W2 + b2
print("shapes:", X.shape, "->", H.shape, "->", out.shape)
print("hidden layer:\n", H)
print("output:", out.ravel())

<div dir="rtl">

### شبكة جاهزة من سايكت-ليرن

</div>

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X, y = make_moons(n_samples=400, noise=0.25, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

logreg = LogisticRegression().fit(X_tr, y_tr)
mlp = MLPClassifier(hidden_layer_sizes=(16, 16), max_iter=2000,
                    random_state=0).fit(X_tr, y_tr)
print(f"logistic regression: {logreg.score(X_te, y_te):.3f}")
print(f"MLP (16, 16)       : {mlp.score(X_te, y_te):.3f}")

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2, 3, 120), np.linspace(-1.6, 2.1, 120))
grid = np.column_stack([xx.ravel(), yy.ravel()])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (name, model) in zip(axes, [("logistic regression", logreg), ("MLP (16, 16)", mlp)]):
    p = model.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=np.linspace(0, 1, 6), cmap="RdBu_r", alpha=0.35)
    ax.contour(xx, yy, p, levels=[0.5], colors="k", linewidths=1.5)
    ax.scatter(X_te[y_te == 0, 0], X_te[y_te == 0, 1], s=18, marker="o")
    ax.scatter(X_te[y_te == 1, 0], X_te[y_te == 1, 1], s=18, marker="s")
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
for i, (W, b) in enumerate(zip(mlp.coefs_, mlp.intercepts_), start=1):
    print(f"layer {i}: W {W.shape}, b {b.shape}")
n_params = sum(W.size + b.size for W, b in zip(mlp.coefs_, mlp.intercepts_))
print("parameters:", n_params)

<div dir="rtl">

## تمرين: احسب شبكة بيدك

شبكة بمدخلين، وطبقة مخفية فيها عصبونان بريلو، ومخرج واحد بدون تفعيل:

$$W_1 = \begin{pmatrix} 1 & -2 \\ 1 & 0.5 \end{pmatrix}, \quad b_1 = (0, 0.5), \quad W_2 = \begin{pmatrix} 2 \\ 4 \end{pmatrix}, \quad b_2 = -1$$

1. احسب المخرج للمدخل $x = (1, 2)$ بيدك، خطوة خطوة.
2. كم معاملاً في شبكة: 784 مدخلاً، ثم 256، ثم 128، ثم 10؟
3. تحقق من الاثنين بالكود.

</div>

In [ ]:
W1 = np.array([[1, -2], [1, 0.5]])
b1 = np.array([0, 0.5])
W2 = np.array([[2], [4]])
b2 = np.array([-1])
x = np.array([1, 2])
# احسب المخرج هنا

sizes = [784, 256, 128, 10]
# عدّ المعاملات هنا